# Customer Purchase Pattern Analyzer
## Comprehensive Exploratory Data Analysis (EDA) & Customer Behavioral Modeling

**Business Vertical:** E-Commerce & Omnichannel Retail  
**Analytical Architecture:** Python 3.12 | Pandas | Plotly | Scikit-Learn | SQLite  
**Story Workflow:**  
`1. Objective` → `2. Data Overview` → `3. Cleaning Summary` → `4. Univariate Analysis` → `5. Bivariate Analysis` → `6. Correlation Matrix` → `7. Trends & Seasonality` → `8. Segmentation` → `9. Strategic Insights` → `10. Recommendations`
---

## 1. Project Objective & Strategic Context

In competitive retail and e-commerce environments, customer acquisition costs continue to rise. Maximizing customer lifetime value (CLV) and understanding purchase cadence, product affinity, and churn dynamics is critical to sustained profitability.

### Key Business Questions Addressed:
1. **Spending Patterns:** How is transaction value distributed, and what is the typical customer basket size?
2. **Customer Loyalty:** What proportion of the customer base makes repeat purchases versus one-off orders?
3. **Demographic Dynamics:** Which age groups generate the highest revenue and highest Average Order Value (AOV)?
4. **Channel & Payment Preferences:** Which payment methods drive the fastest checkout conversion?
5. **Geographic & Category Affinity:** Which product categories dominate across North, South, East, and West regions?
6. **Seasonal Velocity:** How does sales momentum fluctuate across months, particularly during the Q4 Festive Season?
7. **Customer Value Concentration:** How much revenue is driven by top-tier *Champions* versus dormant/at-risk customers?


## 2. Environment Setup & Data Overview

We configure our analytics environment, define consistent corporate visualization tokens using **Plotly**, and load the clean, feature-engineered datasets:
- `data/processed/transactions_features.csv` (Enriched transaction log)
- `data/processed/customer_features.csv` (Aggregated 360° customer profile)


In [1]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Configure Corporate Plotly Styling
pio.templates.default = "plotly_white"
CORPORATE_PALETTE = {
    "primary": "#1E3A8A",       # Deep Navy
    "secondary": "#0D9488",     # Teal
    "accent": "#F59E0B",        # Amber Gold
    "neutral_dark": "#1F2937",  # Charcoal
    "danger": "#EF4444",        # Crimson Red
    "success": "#10B981",       # Emerald Green
    "purple": "#8B5CF6",        # Royal Violet
}

# Load Processed Datasets
trans_path = "../data/processed/transactions_features.csv"
cust_path = "../data/processed/customer_features.csv"

# Fallback to direct paths if running from root
if not os.path.exists(trans_path):
    trans_path = "data/processed/transactions_features.csv"
    cust_path = "data/processed/customer_features.csv"

df_trans = pd.read_csv(trans_path)
df_cust = pd.read_csv(cust_path)

print(f"Transactions Dataset: {df_trans.shape[0]:,} rows, {df_trans.shape[1]} columns")
print(f"Customer Profiles:    {df_cust.shape[0]:,} unique customers, {df_cust.shape[1]} features")
display(df_trans.head(3))
display(df_cust.head(3))


Transactions Dataset: 1,160 rows, 36 columns
Customer Profiles:    129 unique customers, 35 features


,Customer_ID,Customer_Name,Age,Gender,City,Region,Occupation,Product_Category,Product_Name,Purchase_Date,...,Day_of_Week,Is_Weekend,Season,Is_Festive_Season,Season_Festive_Flag,Age_Group,Discount_Amount,Unit_Cost,Estimated_Profit,Margin_Percentage
0,C0106,Manish Bose,62,Male,Bhubaneswar,East,Financial Analyst,Books,Business Strategy Guide,2024-12-22,...,Sunday,True,Winter,True,Festive Peak (Q4),55+,0.00,294.67,884.00,60.00
1,C0132,Arjun Patel,43,Male,Pune,West,Product Manager,Beauty & Personal Care,Herbal Shampoo,2024-10-27,...,Sunday,True,Autumn,True,Festive Peak (Q4),35-44,100.98,235.61,1211.73,63.16
2,C0104,Shreya Bose,65,Male,Kolkata,East,Product Manager,Electronics,Bluetooth Speaker,2024-11-02,...,Saturday,True,Autumn,True,Festive Peak (Q4),55+,325.28,2276.96,650.56,22.22


,Customer_ID,Customer_Name,Age,Age_Group,Gender,City,Region,Occupation,Loyalty_Status,Customer_Segment,...,Basic_CLV,R_Score,F_Score,M_Score,RFM_Score,RFM_Index,RFM_Segment,Cluster,Cluster_Name,Marketing_Action
0,C0001,Rahul Patel,32,25-34,Male,Pune,West,Consultant,Silver,Standard,...,7593.84,1,1,1,111,1.00,Hibernating,1,Lapsed / Low-Frequency Buyers,"Low-cost programmatic re-engagement, seasonal ..."
1,C0002,Manish Joshi,41,35-44,Male,Hyderabad,South,Financial Analyst,Regular,Standard,...,57774.96,5,3,3,533,3.67,Loyal Customers,3,Occasional / Developing Buyers,"Cross-sell bundle recommendations, value-add i..."
2,C0003,Nikhil Verma,47,45-54,Male,Pune,West,Software Engineer,Silver,Standard,...,25242.30,3,2,2,322,2.33,At Risk,3,Occasional / Developing Buyers,Personalized 'We Miss You' win-back email camp...


## 3. Data Cleaning & Pipeline Integrity Summary

Before performing analytics, the raw dataset underwent an 8-step data cleaning pipeline (`src/clean_data.py`):
1. **Deduplication:** 35 exact duplicate records removed.
2. **Name Standardization:** 949 messy customer names stripped of extra whitespace and formatted to Title Case.
3. **Date Harmonization:** Multi-format dates parsed to ISO `YYYY-MM-DD`; 22 corrupted dates flagged.
4. **Taxonomy Cleaning:** 97 category typos resolved using fuzzy matching (`difflib`) and dictionary mappings.
5. **Missing Value Recovery:** Missing totals and unit prices mathematically recomputed ($Total = Qty \times Price \times (1 - Discount)$).
6. **Outlier Treatment:** 82 bulk purchase orders detected using category-specific IQR fences and flagged via `Is_Outlier` (preserving revenue without distortion).
7. **Business Rule Assertions:** 100% passed (positive quantities, date $\le$ today, age between 18 and 80, valid regions).
8. **Customer Alignment:** Purchase frequency and last purchase dates synchronized across all records.

| Metric | Raw Dataset | Processed Clean Dataset | Net Quality Gain |
| :--- | :--- | :--- | :--- |
| **Total Rows** | 1,235 | 1,160 | 75 invalid/duplicate records purged |
| **Missing Values** | 140 | 0 | 100% complete & imputed |
| **Verified Customers** | 130 | 129 | Clean, identifiable cohort |


## 4. Univariate Analysis: Spending & Order Volume

We begin by evaluating individual variable distributions: transaction values and customer order frequencies.


In [2]:
# Chart 1: Transaction Revenue Distribution (Histogram + Box Plot)
fig1 = px.histogram(
    df_trans,
    x="Total_Purchase_Value",
    nbins=50,
    marginal="box",
    title="<b>Distribution of Transaction Revenue (Total Purchase Value)</b>",
    color_discrete_sequence=[CORPORATE_PALETTE["primary"]],
    labels={"Total_Purchase_Value": "Transaction Value (₹)"},
)

fig1.add_vline(
    x=1692.30,
    line_dash="dash",
    line_color=CORPORATE_PALETTE["accent"],
    annotation_text=f"Median: ₹1,692.30",
    annotation_position="top left",
)
fig1.add_vline(
    x=3251.04,
    line_dash="dot",
    line_color=CORPORATE_PALETTE["danger"],
    annotation_text=f"Mean: ₹3,251.04",
    annotation_position="top right",
)

fig1.update_layout(
    xaxis_title="Transaction Value (₹)",
    yaxis_title="Transaction Frequency",
    bargap=0.05,
    font=dict(family="Arial", size=12),
    height=500,
)
fig1.show()


### 🔎 Interpretation of Chart 1: Revenue Distribution
- **Median vs. Mean Asymmetry:** The median transaction value is **₹1,692.30**, whereas the mean is significantly higher at **₹3,251.04**. This right-skewed profile indicates that while the typical consumer order sits between **₹849.80** (25th percentile) and **₹3,392.23** (75th percentile), bulk and high-end transactions pull the financial average upward.
- **Outlier Flagging (82 Transactions):** Exactly **82 orders** exceeded the category-level $1.5 \times IQR$ upper fences (up to ₹172,295.55). By flagging rather than dropping them, the business maintains 100% accounting fidelity for gross revenue of **₹3,771,206.53**.


In [3]:
# Chart 2: Customer Purchase Frequency Distribution
fig2 = px.histogram(
    df_cust,
    x="Order_Count",
    nbins=30,
    title="<b>Customer Purchase Frequency Distribution (Annual Order Count)</b>",
    color_discrete_sequence=[CORPORATE_PALETTE["secondary"]],
    labels={"Order_Count": "Number of Orders per Customer"},
)

fig2.add_vline(
    x=9.0,
    line_dash="dash",
    line_color=CORPORATE_PALETTE["accent"],
    annotation_text=f"Average: 9.0 Orders",
    annotation_position="top right",
)

fig2.update_layout(
    xaxis_title="Lifetime Orders in 2024",
    yaxis_title="Number of Customers",
    bargap=0.1,
    height=450,
)
fig2.show()


### 🔎 Interpretation of Chart 2: Purchase Frequency Distribution
- **Exceptional Repeat Loyalty:** **114 out of 129 customers (88.37%)** are repeat buyers who made 2 or more orders. Only **15 customers (11.63%)** were one-time purchasers.
- **High Purchase Velocity:** The average customer completed **9.0 orders** over the year.
- **Power-Law Long Tail:** A dedicated group of hyper-active buyers completed between 20 and **43 orders** annually, demonstrating that customer retention is the primary growth engine for this retail business.


## 5. Bivariate Analysis: Demographics, Payments & Geography


In [4]:
# Chart 3: Age Group vs. Spend & Average Order Value
age_agg = df_trans.groupby("Age_Group").agg(
    Total_Spend=("Total_Purchase_Value", "sum"),
    Order_Count=("Total_Purchase_Value", "count"),
).reset_index()
age_agg["AOV"] = (age_agg["Total_Spend"] / age_agg["Order_Count"]).round(2)
age_agg["Spend_Share_Pct"] = ((age_agg["Total_Spend"] / age_agg["Total_Spend"].sum()) * 100).round(2)

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=age_agg["Age_Group"],
    y=age_agg["Total_Spend"],
    name="Total Spend (₹)",
    marker_color=CORPORATE_PALETTE["primary"],
    yaxis="y",
    text=[f"₹{v:,.0f}" for v in age_agg["Total_Spend"]],
    textposition="auto",
))

fig3.add_trace(go.Scatter(
    x=age_agg["Age_Group"],
    y=age_agg["AOV"],
    name="Average Order Value (₹)",
    mode="lines+markers+text",
    marker=dict(size=10, color=CORPORATE_PALETTE["accent"]),
    line=dict(width=3, color=CORPORATE_PALETTE["accent"]),
    yaxis="y2",
    text=[f"₹{v:,.0f}" for v in age_agg["AOV"]],
    textposition="top center",
))

fig3.update_layout(
    title="<b>Total Spend and Average Order Value Across Demographic Age Groups</b>",
    xaxis=dict(title="Age Cohort"),
    yaxis=dict(title="Total Spend (₹)", showgrid=True),
    yaxis2=dict(title="Average Order Value (₹)", overlaying="y", side="right", showgrid=False),
    legend=dict(x=0.01, y=0.98, bgcolor="rgba(255,255,255,0.8)"),
    height=480,
)
fig3.show()


### 🔎 Interpretation of Chart 3: Age Group vs. Spend
- **Core Revenue Drivers:** The **35-44** and **45-54** cohorts represent the strongest spending pillars, backed by higher disposable income and steady household re-order requirements.
- **Basket Size Dynamics (AOV):** While young adult consumers (**18-24**) exhibit lower aggregate volumes, selective luxury or tech purchases create competitive AOV figures. Marketing campaigns for high-ticket items should tailor messaging differently for young professionals versus established mature buyers.


In [5]:
# Chart 4: Payment Method Market Share (Donut Chart)
pay_agg = df_trans.groupby("Payment_Method").agg(
    Revenue=("Total_Purchase_Value", "sum"),
    Transactions=("Total_Purchase_Value", "count"),
).reset_index()

fig4 = px.pie(
    pay_agg,
    values="Revenue",
    names="Payment_Method",
    hole=0.45,
    title="<b>Payment Method Revenue Contribution & Channel Adoption</b>",
    color_discrete_sequence=[
        CORPORATE_PALETTE["primary"],
        CORPORATE_PALETTE["secondary"],
        CORPORATE_PALETTE["accent"],
        CORPORATE_PALETTE["purple"],
        "#94A3B8"
    ],
)
fig4.update_traces(textinfo="percent+label", hoverinfo="value+percent")
fig4.update_layout(height=450)
fig4.show()


### 🔎 Interpretation of Chart 4: Payment Method Adoption
- **Dominance of Digital Instant Payments:** **UPI** and **Credit Card** represent over **60% of total revenue**, reflecting seamless checkout flows and zero friction during payment.
- **Low Cash-on-Delivery (COD):** Low COD share indicates high customer trust and minimal return/RTO (Return to Origin) logistics risk, typical of mature digital shoppers.


In [6]:
# Chart 5: Category-by-Region Cross-Tabulation Heatmap
pivot_cat_reg = df_trans.pivot_table(
    index="Region",
    columns="Product_Category",
    values="Total_Purchase_Value",
    aggfunc="sum",
    fill_value=0,
).round(0)

fig5 = px.imshow(
    pivot_cat_reg,
    labels=dict(x="Product Category", y="Geographic Region", color="Revenue (₹)"),
    x=pivot_cat_reg.columns,
    y=pivot_cat_reg.index,
    color_continuous_scale="Blues",
    text_auto=",.0f",
    title="<b>Category Revenue Density by Geographic Region (₹)</b>",
)
fig5.update_layout(height=450)
fig5.show()


### 🔎 Interpretation of Chart 5: Regional Category Density
- **Sports & Fitness Dominates the East:** Customers in the East region generated over **₹343k in Sports & Fitness** purchases alone, representing an extraordinary regional product affinity.
- **Balanced Apparel & Tech in North & West:** **Clothing** and **Electronics** demonstrate balanced, strong demand across the North and West metro hubs (Jaipur, Noida, Pune, Mumbai), suggesting widespread category adoption.


## 6. Correlation Analysis: Customer Attributes & Financial Value


In [7]:
# Chart 6: Correlation Heatmap of Numeric Customer Features
num_cols = [
    "Total_Revenue", "Order_Count", "Average_Order_Value",
    "Total_Quantity", "Recency", "Customer_Tenure",
    "Discount_Usage_Rate", "Basic_CLV"
]
corr_matrix = df_cust[num_cols].corr().round(2)

fig6 = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    range_color=[-1, 1],
    title="<b>Correlation Heatmap of Customer Metrics & Financial Levers</b>",
)
fig6.update_layout(height=520)
fig6.show()


### 🔎 Interpretation of Chart 6: Customer Metric Correlation Matrix
- **Order Count vs. Total Revenue (r = +0.82):** A strong positive correlation confirms that purchase frequency is the single biggest driver of customer lifetime spend, even more so than AOV.
- **Negative Impact of Recency (r = -0.38):** As days since last purchase increase, customer lifetime value declines sharply, emphasizing the vital role of timely re-engagement loops.
- **Discount Usage vs. Revenue (Weak/Neutral correlation):** High discount reliance does not necessarily produce higher revenue; loyalty is driven by product assortment and habit rather than coupon chasing.


## 7. Trends & Seasonality: Trajectory Across 2024


In [8]:
# Chart 7: Monthly Revenue Trend with Festive Season Highlighted
monthly_agg = df_trans.groupby(["Year", "Month", "Month_Name"]).agg(
    Revenue=("Total_Purchase_Value", "sum"),
    Orders=("Total_Purchase_Value", "count"),
).reset_index().sort_values(["Year", "Month"]).reset_index(drop=True)

monthly_agg["MoM_Growth"] = (monthly_agg["Revenue"].pct_change() * 100).round(2).fillna(0.0)

fig7 = go.Figure()

# Base monthly revenue line
fig7.add_trace(go.Bar(
    x=monthly_agg["Month_Name"],
    y=monthly_agg["Revenue"],
    name="Monthly Revenue (₹)",
    marker_color=[
        CORPORATE_PALETTE["danger"] if m in ["October", "November", "December"] 
        else CORPORATE_PALETTE["primary"] 
        for m in monthly_agg["Month_Name"]
    ],
    text=[f"₹{v:,.0f}" for v in monthly_agg["Revenue"]],
    textposition="outside",
))

fig7.add_trace(go.Scatter(
    x=monthly_agg["Month_Name"],
    y=monthly_agg["Revenue"],
    name="Revenue Trend",
    mode="lines+markers",
    line=dict(color=CORPORATE_PALETTE["accent"], width=3),
    marker=dict(size=8),
))

# Highlight Q4 Festive Season Window
fig7.add_vrect(
    x0="October", x1="December",
    fillcolor="rgba(245, 158, 11, 0.15)",
    layer="below", line_width=0,
    annotation_text="<b>Q4 Festive & Holiday Season</b>",
    annotation_position="top left",
)

fig7.update_layout(
    title="<b>Monthly Revenue Trend in 2024 with Q4 Festive Season Highlighted</b>",
    xaxis_title="Month",
    yaxis_title="Total Revenue (₹)",
    height=500,
    yaxis=dict(range=[0, monthly_agg["Revenue"].max() * 1.15]),
)
fig7.show()


### 🔎 Interpretation of Chart 7: Monthly Revenue Trajectory & Seasonality
- **Annual High in April:** **April 2024** achieved the annual peak revenue of **₹476,870.60** (Seasonal Index: 151.7).
- **Q4 Festive Season Rebound:** While October experienced a temporary dip, the festive period accelerated dramatically through November into **December**, surging **+58.55% Month-over-Month** to reach **₹366,331.93**.
- **Year-End Surge:** Year-end clearance and holiday gifting drive high basket sizes and elevated conversion.


## 8. Customer Segmentation Deep Dive: RFM & K-Means Clusters


In [9]:
# Chart 8: Revenue Contribution by RFM Segment
seg_agg = df_cust.groupby("RFM_Segment").agg(
    Revenue=("Total_Revenue", "sum"),
    Customers=("Customer_ID", "count"),
    Avg_CLV=("Basic_CLV", "mean"),
).reset_index().sort_values("Revenue", ascending=False)

fig8 = px.bar(
    seg_agg,
    x="RFM_Segment",
    y="Revenue",
    color="RFM_Segment",
    title="<b>Total Revenue Generated by RFM Customer Segment</b>",
    text=[f"₹{v:,.0f} ({v/seg_agg['Revenue'].sum()*100:.1f}%)" for v in seg_agg["Revenue"]],
    color_discrete_sequence=px.colors.qualitative.Prism,
)
fig8.update_traces(textposition="outside")
fig8.update_layout(xaxis_title="RFM Cohort", yaxis_title="Total Revenue (₹)", showlegend=False, height=480)
fig8.show()


### 🔎 Interpretation of Chart 8: Segment Concentration
- **Champions Drive Majority Value:** **32 Champions** account for **₹2,118,754.10 (56.18% of company revenue)**, with an average CLV of **₹198,633**.
- **Loyal Backing:** The second largest group, **Loyal Customers (26 accounts)**, contributes **21.32%**, meaning top loyalty cohorts produce over **77% of all retail sales**.
- **Intervention Needed for At Risk & Can't Lose Them:** 30 high-value historical customers are currently overdue for re-orders, representing an immediate target for automated win-back workflows.


## 9. Top 5 Executive Strategic Insights

1. **Extreme Revenue Concentration (Pareto Principle):**
   - **40.3% of customers generate 80% of company revenue.** The top 20% alone account for **57.3%** of gross sales.
2. **Frequency Trumps Order Size in CLV:**
   - Transaction frequency ($r = +0.82$ with revenue) is the primary determinant of customer lifetime value. Repeat buyers generate **99.01% of all revenue**.
3. **Regional Product Affinities:**
   - East India displays immense demand for **Sports & Fitness**, while North and West hubs prioritize **Clothing** and **Electronics**.
4. **Digital Checkout Efficiency:**
   - Over **60% of volume flows through instant digital rails (UPI + Credit Cards)**, enabling high-margin transactions with negligible cash handling friction.
5. **Urgent Retention Runway:**
   - **32 repeat customers** have surpassed $1.5\times$ their historical purchase cadence. Re-engaging these buyers before complete dormancy protects over ₹500,000 in annual recurring gross merchandise value.


## 10. Actionable Business Recommendations

| Strategy Pillar | Recommended Initiative | Expected Business Impact |
| :--- | :--- | :--- |
| **VIP Retention** | Launch a tiered VIP Concierge Program for the 32 *Champions* with early access to seasonal lines and dedicated support. | Protect the 56.2% revenue base from competitor poaching. |
| **Automated Win-Back** | Implement automated triggered emails for the 32 overdue repeat buyers offering personalized 20% discount renewal vouchers. | Recapture an estimated ₹150,000–₹250,000 in at-risk revenue. |
| **Cross-Selling Bundles** | Promote high-lift bundles discovered in basket analysis (e.g., *Electric Blender + Power Bank* with 2.24x lift). | Elevate Average Order Value (AOV) by 8–12%. |
| **Regional Inventory Optimization** | Stock higher inventory volumes of Sports & Fitness merchandise in East fulfillment hubs. | Lower fulfillment lead time and boost regional conversion. |
| **Onboarding Nurture** | Provide a time-limited 15% discount on second orders for the 5 *New Customers*. | Accelerate transition from one-time trial to long-term repeat loyalty. |

---
*Notebook created and validated end-to-end for Customer Purchase Pattern Analyzer.*
